# 023 — GraphRAG (Knowledge Graph RAG)
**Advanced RAG Series** | Entity-based retrieval using knowledge graphs

Covers: Entity extraction · Graph construction · Graph traversal · Graph-augmented answers


In [1]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=".*LangChain.*")
warnings.filterwarnings('ignore', module="langchain.*")
warnings.filterwarnings('ignore', module="langgraph.*")

from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("✓ API key loaded")


✓ API key loaded


In [2]:
import os, time
os.environ["ANONYMIZED_TELEMETRY"] = "False"

from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=2)
llm_smart = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2)
print(f"✓ llm       : {llm.model_name}")
print(f"✓ llm_smart : {llm_smart.model_name}")


✓ llm       : llama-3.1-8b-instant
✓ llm_smart : llama-3.3-70b-versatile


In [3]:
import os
DS_DIR = os.path.join(os.getcwd(), '..', 'datasets')

def load_doc(fname):
    path = os.path.join(DS_DIR, fname)
    if os.path.exists(path):
        return open(path, encoding='utf-8').read()
    raise FileNotFoundError(f"Missing: {path}\nRun python create_datasets.py from project root.")

ai_text  = load_doc("ai.txt")
ml_text  = load_doc("machine_learning.txt")
nlp_text = load_doc("nlp.txt")
rag_text = load_doc("rag.txt")
all_text = "\n\n".join([ai_text, ml_text, nlp_text, rag_text])
print(f"✓ Loaded {len(all_text):,} chars across 4 files")


✓ Loaded 22,486 chars across 4 files


In [4]:
# ── 1. What is GraphRAG? ────────────────────────────────────────────────
print("STANDARD RAG: chunk documents → embed → retrieve by vector similarity")
print("GRAPHRAG    : extract entities + relationships → build graph →")
print("              retrieve by graph traversal + community detection")
print()
print("Key advantage: captures RELATIONSHIPS between concepts,")
print("not just similarity of isolated chunks.")
print()
print("Example:")
print("  'BERT was created by Google and uses the Transformer architecture'")
print("  Standard RAG: stores this as a chunk")
print("  GraphRAG:     stores: BERT --created_by--> Google")
print("                         BERT --uses--> Transformer")


STANDARD RAG: chunk documents → embed → retrieve by vector similarity
GRAPHRAG    : extract entities + relationships → build graph →
              retrieve by graph traversal + community detection

Key advantage: captures RELATIONSHIPS between concepts,
not just similarity of isolated chunks.

Example:
  'BERT was created by Google and uses the Transformer architecture'
  Standard RAG: stores this as a chunk
  GraphRAG:     stores: BERT --created_by--> Google
                         BERT --uses--> Transformer


In [5]:
# ── 2. Extract entities and relationships with LLM ──────────────────────
import json

EXTRACT_PROMPT = (
    "Extract entities and relationships from the text below.\n"
    "Return JSON with this structure:\n"
    '{{"entities": ["Entity1", "Entity2", ...], '
    '"relationships": [["Entity1", "relation", "Entity2"], ...]}}\n'
    "Use short entity names. Relationships should be 1-3 words.\n"
    "Text:\n{text}\n\n"
    "JSON:"
)

def extract_graph(text: str) -> dict:
    response = llm.invoke(EXTRACT_PROMPT.format(text=text))
    raw = response.content.strip()
    # Strip markdown code fences if present
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    try:
        return json.loads(raw.strip())
    except json.JSONDecodeError:
        return {"entities": [], "relationships": []}

# Test on a paragraph
sample = rag_text[:800]
result = extract_graph(sample)
print(f"Entities found: {result['entities'][:10]}")
print(f"\nRelationships found:")
for r in result['relationships'][:8]:
    print(f"  {r[0]} --{r[1]}--> {r[2]}")


Entities found: []

Relationships found:


In [6]:
# ── 3. Build knowledge graph from multiple chunks ───────────────────────
import networkx as nx
from langchain_text_splitters import RecursiveCharacterTextSplitter
import time

splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=60)
chunks = splitter.split_text(rag_text)[:10]  # 10 chunks for demo

G = nx.DiGraph()
chunk_map = {}  # entity → list of source chunks

print(f"Building graph from {len(chunks)} chunks...")
for i, chunk in enumerate(chunks):
    data = extract_graph(chunk)
    for entity in data['entities']:
        entity = entity.strip()
        if not G.has_node(entity):
            G.add_node(entity)
        chunk_map.setdefault(entity, []).append(chunk)
    for rel in data['relationships']:
        if len(rel) == 3:
            src, relation, tgt = rel[0].strip(), rel[1].strip(), rel[2].strip()
            if src and tgt:
                G.add_edge(src, tgt, relation=relation)
    time.sleep(0.3)

print(f"\n✓ Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")


Building graph from 10 chunks...



✓ Graph: 43 nodes, 25 edges


In [7]:
# ── 4. Inspect the graph ────────────────────────────────────────────────
print("Top entities by degree (most connected):")
degree_sorted = sorted(G.degree(), key=lambda x: x[1], reverse=True)
for node, deg in degree_sorted[:10]:
    print(f"  {node:<30} degree={deg}")

print("\nSample edges (entity relationships):")
for src, tgt, data in list(G.edges(data=True))[:10]:
    print(f"  {src} --{data.get('relation', '?')}--> {tgt}")


Top entities by degree (most connected):
  documents                      degree=5
  BM25                           degree=4
  RAG                            degree=3
  Dense Retrieval                degree=3
  HyDE                           degree=3
  Self-RAG                       degree=3
  CRAG                           degree=3
  RAPTOR                         degree=3
  retrieval                      degree=2
  context                        degree=2

Sample edges (entity relationships):
  RAG --retrieves--> documents
  RAG --uses--> context
  RAG --generates--> responses
  Dense Retrieval --uses--> Embedding Similarity
  Dense Retrieval --uses--> Bi-encoder
  Dense Retrieval --uses--> All-MiniLM-L6-v2
  Sparse Retrieval --uses--> Term Frequency
  Sparse Retrieval --uses--> BM25
  BM25 --scores--> documents
  BM25 --excels--> exact keyword matching


In [8]:
# ── 5. Graph-based retrieval ────────────────────────────────────────────
def extract_query_entities(query: str) -> list:
    response = llm.invoke(
        f"List the key entities in this question (comma-separated, no explanation):\n{query}"
    )
    return [e.strip() for e in response.content.split(",")]

def graph_retrieve(query: str, hops: int = 2) -> list:
    # Find query entities in graph
    q_entities = extract_query_entities(query)
    found = [e for e in q_entities if G.has_node(e)]

    # Collect neighbors up to `hops` away
    relevant_entities = set(found)
    for entity in found:
        for node in nx.ego_graph(G, entity, radius=hops).nodes():
            relevant_entities.add(node)

    # Gather source chunks for relevant entities
    relevant_chunks = []
    seen = set()
    for entity in relevant_entities:
        for chunk in chunk_map.get(entity, []):
            if chunk[:80] not in seen:
                seen.add(chunk[:80])
                relevant_chunks.append(chunk)

    return relevant_chunks[:5]

query = "How does RAG use vector embeddings?"
retrieved = graph_retrieve(query)
print(f"Query: {query!r}")
print(f"Graph-retrieved {len(retrieved)} chunks")
if retrieved:
    print("\nTop chunk:")
    print(retrieved[0][:300])


Query: 'How does RAG use vector embeddings?'
Graph-retrieved 2 chunks

Top chunk:
Retrieval-Augmented Generation (RAG)

Retrieval-Augmented Generation (RAG) is a hybrid AI architecture that combines information
retrieval with text generation. Rather than relying solely on knowledge stored in model
parameters, RAG systems retrieve relevant documents from an external knowledge base


In [9]:
# ── 6. Full GraphRAG answer ─────────────────────────────────────────────
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = ChatPromptTemplate.from_template(
    "You are a helpful assistant. Answer ONLY using the context below.\n"
    "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
)

def graphrag_answer(question: str) -> str:
    chunks_r = graph_retrieve(question)
    if not chunks_r:
        return "No relevant graph context found."
    context = "\n\n---\n\n".join(chunks_r)
    chain = RAG_PROMPT | llm
    return chain.invoke({"context": context, "question": question}).content

answer = graphrag_answer("What is the relationship between RAG and vector databases?")
print(answer)


There is no information provided in the given context about the relationship between RAG and vector databases.


In [10]:
# ── 7. GraphRAG vs Standard RAG summary ─────────────────────────────────
print("GRAPHRAG vs STANDARD RAG")
print("=" * 45)
print()
print("Standard RAG:")
print("  Retrieval = semantic similarity of isolated chunks")
print("  Misses: multi-hop relationships between concepts")
print()
print("GraphRAG:")
print("  Retrieval = graph traversal from query entities")
print("  Captures: who created what, how X relates to Y")
print("  Best for: connected knowledge domains (research, legal, medical)")
print()
print("Cost:")
print("  Index time: N LLM calls for entity extraction (one per chunk)")
print("  Query time: 1 LLM call to extract query entities + graph lookup")
print()
print("Microsoft GraphRAG (2024):")
print("  Adds community detection (Leiden algorithm)")
print("  Generates community summaries for global queries")
print("  Available as: pip install graphrag")


GRAPHRAG vs STANDARD RAG

Standard RAG:
  Retrieval = semantic similarity of isolated chunks
  Misses: multi-hop relationships between concepts

GraphRAG:
  Retrieval = graph traversal from query entities
  Captures: who created what, how X relates to Y
  Best for: connected knowledge domains (research, legal, medical)

Cost:
  Index time: N LLM calls for entity extraction (one per chunk)
  Query time: 1 LLM call to extract query entities + graph lookup

Microsoft GraphRAG (2024):
  Adds community detection (Leiden algorithm)
  Generates community summaries for global queries
  Available as: pip install graphrag
